# PE6201 A2 — Final Demo
**Group B-3**

**September 2026**

**Problem B: Outpatient Referral Coordination**

The notebook expects:

`/content/drive/MyDrive/PE6201_A2/A2_scaffold/`


## SETUP — run before recording

This cell mounts Google Drive and points the notebook to the final submission scaffold.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, statistics, importlib

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

PROJECT_ROOT = Path("/content/drive/MyDrive/PE6201_A2")
SCAFFOLD = PROJECT_ROOT / "A2_scaffold"
REFERENCE_DATA = PROJECT_ROOT / "A2_reference_data"
RESULTS = SCAFFOLD / "results"

os.environ["A2_DATA"] = str(REFERENCE_DATA)

if str(SCAFFOLD) not in sys.path:
    sys.path.insert(0, str(SCAFFOLD))

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SCAFFOLD     :", SCAFFOLD)
print("RESULTS      :", RESULTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT : /content/drive/MyDrive/PE6201_A2
SCAFFOLD     : /content/drive/MyDrive/PE6201_A2/A2_scaffold
RESULTS      : /content/drive/MyDrive/PE6201_A2/A2_scaffold/results


## Helper functions — run before recording

These functions only format or execute the final submitted system. No live-model battery is rerun.

In [7]:
import py_compile
import contextlib
import io
import subprocess
import shutil
import importlib.util

CORE_FILES = [
    "agent.py", "backends.py", "config.py", "guardrails.py",
    "harness.py", "prompt.py", "run_eval.py", "tools.py", "tools_ori.py",
    "test_guardrails.py", "agent_broken.py", "tools_broken.py",
    "run_failure1.py", "run_failure2.py",
]

REQUIRED_RESULTS = [
    "d2a_tool_block_comparison.json",
    "d2b_comparison_summary.json",
    "d2c_comparison_summary.json",
    "d3_guardrail_summary.json",
    "d4_scripted_56_summary.json",
    "d4_human_review_edited.json",
    "d7_failure1_step_cap.json",
    "d7_failure2_interface_band.json",
    "D6 Cost-to-Serve Analysis.pdf",
]

def load_json(name):
    p = RESULTS / name
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_project_modules():
    for name in [
        "config", "tools", "prompt", "guardrails",
        "backends", "agent", "harness", "test_guardrails"
    ]:
        sys.modules.pop(name, None)

def load_project(mode="parallel"):
    """Fresh-load the submitted project in free scripted mode."""
    os.environ["A2_DATA"] = str(REFERENCE_DATA)
    os.chdir(SCAFFOLD)
    if str(SCAFFOLD) not in sys.path:
        sys.path.insert(0, str(SCAFFOLD))

    clean_project_modules()

    import config
    config.BACKEND = "scripted"
    config.VERSION = "v2"
    config.TOOL_CALL_MODE = mode

    import tools
    import prompt
    import backends
    import guardrails
    import agent
    import harness

    return config, tools, prompt, backends, guardrails, agent, harness

def load_tools_ori():
    """Load the original lecturer/scaffold tool file without replacing final tools.py."""
    ori_path = SCAFFOLD / "tools_ori.py"
    if not ori_path.exists():
        raise FileNotFoundError(f"Missing {ori_path}")
    if "config" not in sys.modules:
        load_project("parallel")
    spec = importlib.util.spec_from_file_location("tools_ori_demo", ori_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules["tools_ori_demo"] = module
    spec.loader.exec_module(module)
    return module

def preflight():
    print("=" * 72)
    print("FINAL SUBMISSION PREFLIGHT")
    print("=" * 72)

    ok = True
    for filename in CORE_FILES:
        p = SCAFFOLD / filename
        if not p.exists():
            print(f"FAIL  {filename} missing")
            ok = False
            continue
        try:
            py_compile.compile(str(p), doraise=True)
            print(f"PASS  {filename}")
        except Exception as e:
            print(f"FAIL  {filename}: {e}")
            ok = False

    print("\nResult evidence:")
    for filename in REQUIRED_RESULTS:
        exists = (RESULTS / filename).exists()
        print(f"{'PASS' if exists else 'FAIL'}  {filename}")
        ok = ok and exists

    d5_files = sorted(RESULTS.glob("d5_*.json"))
    print(f"\nD5 result files found: {len(d5_files)}")
    for p in d5_files:
        print("  ", p.name)

    os.chdir(SCAFFOLD)
    clean_project_modules()
    import config
    print("\nDefault backend:", config.BACKEND)
    if config.BACKEND != "scripted":
        print("WARNING: final submitted config should default to scripted.")
        ok = False

    print("\nPREFLIGHT:", "PASS" if ok else "CHECK ITEMS ABOVE")
    return ok

def run_case_demo(case_id, verbose=True):
    """Show the evaluation case first, then actually run and grade it."""
    config, tools, prompt, backends, guardrails, agent, harness = (
        load_project("parallel")
    )

    # ================================================================
    # 1. SHOW THE EVALUATION CASE / GROUND TRUTH FIRST
    # ================================================================
    key = harness.load_key("B")

    if case_id not in key:
        raise KeyError(f"{case_id} not found in expected_outcomes_B.json")

    expected = key[case_id]

    # Load the actual referral record used by this evaluation case
    referrals_path = (
        REFERENCE_DATA / "data_B" / "referrals.json"
    )

    with open(referrals_path, "r", encoding="utf-8") as f:
        referrals = json.load(f)

    referral = next(
        (
            r for r in referrals
            if r.get("referral_id") == case_id
        ),
        None
    )

    print("\n" + "=" * 72)
    print("EVALUATION CASE")
    print("=" * 72)

    print("Case ID             :", case_id)

    if referral:
        print("Patient ID          :", referral.get("patient_id"))
        print("Specialty           :", referral.get("specialty"))
        print("Date received       :", referral.get("date_received"))
        print("Clinical summary    :", referral.get("clinical_summary"))
        print(
            "Tests attached      :",
            referral.get("tests_attached", [])
        )

    print("\nEXPECTED OUTCOME")
    print("Expected decision   :", expected.get("expected_decision"))

    if expected.get("trigger"):
        print("Expected trigger    :", expected.get("trigger"))

    if expected.get("missing"):
        print("Expected missing    :", expected.get("missing"))

    if expected.get("booked"):
        booked = expected["booked"]
        print(
            "Expected booking    :",
            f"{booked.get('clinic')} | "
            f"{booked.get('date')} | "
            f"{booked.get('time')}"
        )

    if expected.get("must_record"):
        print(
            "Must record         :",
            "; ".join(expected.get("must_record", []))
        )

    print("\n" + "-" * 72)
    print("NOW RUNNING THE AGENT")
    print("-" * 72)

    # ================================================================
    # 2. ACTUALLY RUN THE AGENT
    # ================================================================
    record = agent.run_case(
        case_id,
        problem="B",
        approve=lambda action, payload: True,
        verbose=verbose,
    )

    # ================================================================
    # 3. GRADE AGAINST THE SAME EVALUATION CASE
    # ================================================================
    passed, fails = harness.code_check(
        record,
        expected
    )

    print("\n" + "=" * 72)
    print("CASE RESULT")
    print("=" * 72)

    print("Case       :", case_id)
    print("Decision   :", record.get("decision"))

    if record.get("trigger"):
        print("Trigger    :", record.get("trigger"))

    if record.get("missing"):
        print("Missing    :", record.get("missing"))

    if record.get("booked"):
        booked = record.get("booked")
        print(
            "Booked     :",
            f"{booked.get('clinic')} | "
            f"{booked.get('date')} | "
            f"{booked.get('time')}"
        )

    print("Turns      :", record.get("turns"))

    print(
        "Tool calls :",
        record.get(
            "tool_calls",
            len(record.get("evidence", []))
        )
    )

    print(
        "Evidence   :",
        " → ".join(record.get("evidence", []))
    )

    print(
        "Code check :",
        "PASS" if passed else "FAIL"
    )

    if fails:
        print("Failures   :", fails)

    print("=" * 72 + "\n")

    return record, passed, fails



# ---------------------------------------------------------------------
# D2 — DETAILED ACTUAL RE-EXECUTION
# ---------------------------------------------------------------------

def run_d2a_detailed():
    print("=" * 76)
    print("D2(a) — TOOL SET MINIMISATION")
    print("=" * 76)

    config, tools, prompt, backends, guardrails, agent, harness = (
        load_project("parallel")
    )
    tools_ori = load_tools_ori()

    before_names = list(tools_ori.REGISTRY["B"])
    after_names = list(tools.REGISTRY["B"])

    def show_tool_difference():
        """Show the actual Problem B tool-set difference before and after."""

        before_set = set(before_names)
        after_set = set(after_names)

        removed = sorted(before_set - after_set)
        added = sorted(after_set - before_set)
        kept = sorted(before_set & after_set)

        print("\n" + "=" * 76)
        print("D2(a) — TOOL SET DIFFERENCE")
        print("=" * 76)

        print("\nORIGINAL — tools_ori.py")
        for i, name in enumerate(before_names, 1):
            print(f"  {i}. {name}")

        print("\nFINAL — tools.py")
        for i, name in enumerate(after_names, 1):
            print(f"  {i}. {name}")

        print("\nCHANGES")
        print("  Removed :", ", ".join(removed) if removed else "None")
        print("  Added   :", ", ".join(added) if added else "None")
        print("  Kept    :", ", ".join(kept))

        print("\nWHY")
        if "as_of" in removed:
            print(
                "  as_of was removed from the model-facing tool set. "
                "Its date/reference-window logic is handled inside "
                "check_referral_criteria, so the model does not need a "
                "separate call just to obtain the reference date."
            )

        print(
            "\nRESULT: "
            f"{len(before_names)} model-facing tools "
            f"→ {len(after_names)} model-facing tools"
        )

    show_tool_difference()

    def descriptor_text(module, names):
        blocks = []
        for name in names:
            if name in module.DESCRIPTORS:
                blocks.append(json.dumps(module.DESCRIPTORS[name], ensure_ascii=False,
                                         sort_keys=True, separators=(",", ":")))
        return "\n".join(blocks)

    before_text = descriptor_text(tools_ori, before_names)
    after_text = descriptor_text(tools, after_names)
    before_chars, after_chars = len(before_text), len(after_text)
    before_tokens, after_tokens = before_chars / 4, after_chars / 4

    print("\n" + "-" * 76)
    print("BEFORE — tools_ori.py")
    print("-" * 76)
    print(f"Callable Problem B tools: {len(before_names)}")
    for i, name in enumerate(before_names, 1):
        print(f"\n[{i}] {name}")
        if name in tools_ori.DESCRIPTORS:
            print(json.dumps(tools_ori.DESCRIPTORS[name], indent=2, ensure_ascii=False))
        else:
            print("  No descriptor")
    print(f"\nBEFORE descriptor size: {before_chars:,} chars ≈ {before_tokens:.0f} tokens")

    print("\n" + "-" * 76)
    print("AFTER — final tools.py")
    print("-" * 76)
    print(f"Callable Problem B tools: {len(after_names)}")
    for i, name in enumerate(after_names, 1):
        print(f"\n[{i}] {name}")
        if name in tools.DESCRIPTORS:
            print(json.dumps(tools.DESCRIPTORS[name], indent=2, ensure_ascii=False))
        else:
            print("  No descriptor")
    print(f"\nAFTER descriptor size: {after_chars:,} chars ≈ {after_tokens:.0f} tokens")

    removed = [x for x in before_names if x not in after_names]
    added = [x for x in after_names if x not in before_names]
    reduction = before_tokens - after_tokens
    reduction_pct = reduction / before_tokens * 100 if before_tokens else 0

    print("\n" + "=" * 76)
    print("D2(a) SUMMARY")
    print("=" * 76)
    print(f"Callable tools : {len(before_names)} → {len(after_names)}")
    print("Removed        :", ", ".join(removed) if removed else "None")
    print("Added          :", ", ".join(added) if added else "None")
    print(f"Approx tokens  : {before_tokens:.0f} → {after_tokens:.0f}")
    print(f"Reduction      : {reduction:.0f} tokens ({reduction_pct:.2f}%)")
    print("Method         : descriptor characters / 4")

    archived = load_json("d2a_tool_block_comparison.json")
    print("Frozen isolated experiment:",
          f"{archived['before']['approx_descriptor_tokens_chars_div_4']} → "
          f"{archived['after']['approx_descriptor_tokens_chars_div_4']} tokens "
          f"({archived['approx_percent_reduction']}%)")

    return {"before_tools": before_names, "after_tools": after_names,
            "before_tokens": before_tokens, "after_tokens": after_tokens,
            "reduction_percent": reduction_pct}


def get_lecturer_cases():
    """Fixture cases not created by our team (i.e. not REF-EV001..040)."""
    config, tools, prompt, backends, guardrails, agent, harness = load_project("parallel")
    all_cases = harness.load_cases()
    team_cases = {f"REF-EV{i:03d}" for i in range(1, 41)}
    return [cid for cid in all_cases if cid not in team_cases]


def run_d2b_detailed():
    """Run lecturer/shipped fixture cases through both v1 and v2 slot interfaces."""
    config, tools, prompt, backends, guardrails, agent, harness = load_project("parallel")

    def show_get_clinic_slots_difference():
        print("\n" + "=" * 76)
        print("D2(b) — get_clinic_slots V1 vs V2")
        print("=" * 76)

        print("\nV1")
        print("  Signature : get_clinic_slots(specialty, from, to)")
        print("  Band      : NOT required")
        print("  Filtering : specialty + date + capacity")
        print("  Risk      : wrong-band slots can still be returned")

        print("\nV2")
        print("  Signature : get_clinic_slots(specialty, band, from, to)")
        print("  Band      : REQUIRED")
        print("  Filtering : specialty + band + date + capacity")
        print("  Result    : wrong-band rows are filtered out inside the tool")

        print("\nPOKA-YOKE EFFECT")
        print(
            "  The model can no longer omit urgency band and accidentally "
            "receive slots from another band."
        )

    show_get_clinic_slots_difference()

    lecturer_cases = get_lecturer_cases()

    print("=" * 76)
    print("D2(b) — V1 vs V2 TOOL INTERFACE")
    print("=" * 76)
    print("Lecturer/shipped fixture cases detected:", lecturer_cases)

    details = []
    for cid in lecturer_cases:
        ref = tools.get_referral(cid)
        if not ref:
            continue
        specialty = ref["specialty"]
        criteria = tools.check_referral_criteria(specialty=specialty, referral_id=cid)
        if not criteria:
            continue
        if criteria.get("red_flag_term") or not criteria.get("right_department", True) or criteria.get("missing_tests"):
            continue

        band = criteria["band"]
        window_start = criteria.get("window_start")
        window_end = criteria.get("window_end")
        if not window_start or not window_end:
            continue

        v1 = tools._get_clinic_slots_v1(
            specialty=specialty,
            **{"from": window_start, "to": window_end}
        )
        v2 = tools._get_clinic_slots_v2(
            specialty=specialty, band=band,
            **{"from": window_start, "to": window_end}
        )

        v1_json = json.dumps(v1, ensure_ascii=False, separators=(",", ":"))
        v2_json = json.dumps(v2, ensure_ascii=False, separators=(",", ":"))
        v1_tokens, v2_tokens = len(v1_json)/4, len(v2_json)/4

        details.append({"case_id": cid, "specialty": specialty, "band": band,
                        "v1_rows": len(v1), "v2_rows": len(v2),
                        "v1_tokens": v1_tokens, "v2_tokens": v2_tokens})

        print("\n" + "-" * 76)
        print(f"CASE {cid}")
        print("-" * 76)
        print(f"Specialty={specialty} | Band={band} | Window={window_start} → {window_end}")
        print("\nV1 RESULT")
        print(json.dumps(v1, indent=2))
        print(f"Rows={len(v1)} | Approx observation tokens={v1_tokens:.2f}")
        print("\nV2 RESULT")
        print(json.dumps(v2, indent=2))
        print(f"Rows={len(v2)} | Approx observation tokens={v2_tokens:.2f}")

    archived = load_json("d2b_comparison_summary.json")
    obs = archived["observation_comparison"]
    print("\n" + "=" * 76)
    print("D2(b) FULL COMPARISON SUMMARY")
    print("=" * 76)
    print(f"Cases measured       : {obs['eligible_slot_cases_measured']}")
    print(f"Tool calls           : {obs['eligible_slot_cases_measured']} v1 / {obs['eligible_slot_cases_measured']} v2")
    print(f"Mean tokens/call     : {obs['v1_mean_observation_approx_tokens']:.2f} → {obs['v2_mean_observation_approx_tokens']:.2f}")
    print(f"Total obs. tokens    : {obs['v1_total_observation_approx_tokens']:,} → {obs['v2_total_observation_approx_tokens']:,}")
    print(f"Mean rows/call       : {obs['v1_mean_rows_returned']:.2f} → {obs['v2_mean_rows_returned']:.2f}")
    print(f"Observation reduction: {abs(obs['mean_observation_percent_change_v2_vs_v1']):.2f}%")
    print(f"Full prompt tokens   : {archived['v1']['full_prompt_approx_tokens']} → {archived['v2']['full_prompt_approx_tokens']}")
    return details


def run_d2c_detailed():

    def show_serial_parallel_difference():
        print("\n" + "=" * 76)
        print("D2(c) — SERIAL vs PARALLEL TOOL CALLING")
        print("=" * 76)

        print("\nSERIAL")
        print("  Rule      : one tool call per model turn")
        print("  Example   :")
        print("    Turn 1 -> get_referral")
        print("    Turn 2 -> check_referral_criteria")
        print("    Turn 3 -> lookup_patient")
        print("    Turn 4 -> get_clinic_slots")
        print("    Turn 5 -> book_slot")

        print("\nPARALLEL")
        print("  Rule      : independent calls may share the same model turn")
        print("  Example   :")
        print("    Turn 1 -> get_referral")
        print("    Turn 2 -> check_referral_criteria || lookup_patient")
        print("    Turn 3 -> get_clinic_slots")
        print("    Turn 4 -> book_slot")

        print("\nDEPENDENCY RULE")
        print(
            "  Calls are parallelised only when neither call needs "
            "the other call's output."
        )
        print(
            "  get_clinic_slots stays sequential because it needs the "
            "urgency band and booking window from check_referral_criteria."
        )

        print("\nEXPECTED EFFECT")
        print("  Same tool work, fewer model turns, therefore fewer repeated")
        print("  prompt/history tokens and lower cost if correctness is unchanged.")

    show_serial_parallel_difference()

    """Print all 40 serial results, all 40 parallel results, then aggregate summary."""
    def run_mode(mode):
        config, tools, prompt, backends, guardrails, agent, harness = load_project(mode)
        cases = [f"REF-EV{i:03d}" for i in range(1, 41)]
        hidden = io.StringIO()
        with contextlib.redirect_stdout(hidden):
            results, _ = harness.run_set(cases, problem="B",
                                         trials_for=lambda cid: 1,
                                         verbose=False)
        return results

    serial = run_mode("serial")
    parallel = run_mode("parallel")

    print("=" * 76)
    print("D2(c) — SEQUENTIAL RESULTS")
    print("=" * 76)
    for r in serial:
        rec = r["record"]
        print(f"{r['case_id']:<12} {'PASS' if r['passed'] else 'FAIL':<5} "
              f"decision={str(rec.get('decision')):<20} turns={rec.get('turns'):<2} "
              f"calls={rec.get('tool_calls'):<2} in={rec.get('tokens_in'):<7} "
              f"out={rec.get('tokens_out'):<5} cost=${rec.get('cost_usd'):.6f}")

    print("\n" + "=" * 76)
    print("D2(c) — PARALLEL RESULTS")
    print("=" * 76)
    for r in parallel:
        rec = r["record"]
        print(f"{r['case_id']:<12} {'PASS' if r['passed'] else 'FAIL':<5} "
              f"decision={str(rec.get('decision')):<20} turns={rec.get('turns'):<2} "
              f"calls={rec.get('tool_calls'):<2} in={rec.get('tokens_in'):<7} "
              f"out={rec.get('tokens_out'):<5} cost=${rec.get('cost_usd'):.6f}")

    def summarise(results):
        records = [r["record"] for r in results]
        turns = [x["turns"] for x in records]
        return {
            "cases": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "mean_turns": sum(turns)/len(turns),
            "median_turns": statistics.median(turns),
            "tool_calls": sum(x.get("tool_calls",0) for x in records),
            "tokens_in": sum(x.get("tokens_in",0) for x in records),
            "tokens_out": sum(x.get("tokens_out",0) for x in records),
            "cost": sum(x.get("cost_usd",0) for x in records),
        }

    s, p = summarise(serial), summarise(parallel)
    reduction = (s["mean_turns"] - p["mean_turns"]) / s["mean_turns"] * 100

    print("\n" + "=" * 76)
    print("D2(c) SUMMARY")
    print("=" * 76)
    print(f"Pass rate     : {s['passed']}/{s['cases']} → {p['passed']}/{p['cases']}")
    print(f"Mean turns    : {s['mean_turns']:.3f} → {p['mean_turns']:.3f}")
    print(f"Median turns  : {s['median_turns']:.0f} → {p['median_turns']:.0f}")
    print(f"Tool calls    : {s['tool_calls']} → {p['tool_calls']}")
    print(f"Input tokens  : {s['tokens_in']:,} → {p['tokens_in']:,}")
    print(f"Output tokens : {s['tokens_out']:,} → {p['tokens_out']:,}")
    print(f"Cost          : ${s['cost']:.5f} → ${p['cost']:.5f}")
    print(f"Mean-turn reduction: {reduction:.2f}%")
    return {"serial": serial, "parallel": parallel, "summary": {"serial": s, "parallel": p}}


# ---------------------------------------------------------------------
# D3 + D4 — DETAILED ACTUAL RE-EXECUTION
# ---------------------------------------------------------------------

def run_d3_actual():
    config, tools, prompt, backends, guardrails, agent, harness = load_project("parallel")
    sys.modules.pop("test_guardrails", None)
    import test_guardrails as tg

    tests = [
        tg.test_gr01_step_cap_excessive_turns,
        tg.test_gr02_budget_ceiling_token_limit,
        tg.test_gr03_duplicate_action_loop_detection,
        tg.test_gr04_autonomy_gate_suggest_mode,
        tg.test_gr05_autonomy_gate_confirm_mode_denied,
        tg.test_gr06_step_cap_just_under_limit,
        tg.test_gr07_duplicate_different_args,
        tg.test_gr08_hostile_clinical_summary_duplicate_action,
        tg.test_gr09_hostile_clinical_summary_loop,
        tg.test_gr10_hostile_clinical_summary_token_explosion,
        tg.test_gr11_parallel_duplicate_calls,
        tg.test_gr12_budget_ceiling_cumulative,
        tg.test_gr13_gate_before_book_slot,
        tg.test_gr14_autonomy_act_mode,
    ]
    hidden = io.StringIO()
    with contextlib.redirect_stdout(hidden):
        results = [fn() for fn in tests]
    passed = sum(1 for r in results if r.passed)
    hostile = [r for r in results if r.test_id in {"GR-08","GR-09","GR-10"}]
    hostile_passed = sum(1 for r in hostile if r.passed)

    print("=" * 76)
    print("D3 — GUARDRAIL RESULTS")
    print("=" * 76)
    for r in results:
        print(f"{'PASS' if r.passed else 'FAIL'} {r.test_id:<6} {r.actual_guardrail or 'allowed'}")
    print("\nD3 SUMMARY")
    print(f"All tests    : {passed}/{len(results)} PASS")
    print(f"Hostile text : {hostile_passed}/{len(hostile)} PASS")
    return results


def run_d4_detailed():
    """Show 40 unique cases, then explain the 56-trial battery and summary."""
    config, tools, prompt, backends, guardrails, agent, harness = load_project("parallel")
    cases = [f"REF-EV{i:03d}" for i in range(1, 41)]
    negative_ids = {f"REF-EV{i:03d}" for i in range(33, 41)}

    hidden = io.StringIO()
    with contextlib.redirect_stdout(hidden):
        results, _ = harness.run_set(cases, problem="B", verbose=False)

    first_result = {}
    for r in results:
        first_result.setdefault(r["case_id"], r)

    print("=" * 76)
    print("D4 — 40 UNIQUE EVALUATION CASES")
    print("=" * 76)
    for cid in cases:
        r = first_result[cid]
        rec = r["record"]
        marker = "NEGATIVE" if cid in negative_ids else "ordinary"
        print(f"{cid:<12} {marker:<8} {'PASS' if r['passed'] else 'FAIL':<5} "
              f"decision={str(rec.get('decision')):<20} turns={rec.get('turns')}")

    print("\n" + "=" * 76)
    print("WHY 40 CASES BECOME 56 TRIALS")
    print("=" * 76)
    print("Ordinary cases: REF-EV001 … REF-EV032")
    print("32 cases × 1 trial = 32 trials")
    print("\nNegative cases repeated 3 times each:")
    for cid in sorted(negative_ids):
        matching = [r for r in results if r["case_id"] == cid]
        passed_n = sum(1 for r in matching if r["passed"])
        print(f"  {cid}: {passed_n}/{len(matching)} PASS")
    print("\n8 negative cases × 3 trials = 24 trials")
    print("32 ordinary + 24 negative = 56 total trials")

    passed = sum(1 for r in results if r["passed"])
    negative_results = [r for r in results if r["case_id"] in negative_ids]
    negative_passed = sum(1 for r in negative_results if r["passed"])
    turns = [r["record"]["turns"] for r in results]

    print("\n" + "=" * 76)
    print("D4 SUMMARY")
    print("=" * 76)
    print("Unique cases       : 40")
    print("Ordinary cases     : 32")
    print("Negative cases     : 8")
    print(f"Total trials       : {len(results)}")
    print(f"Overall pass       : {passed}/{len(results)} ({passed/len(results)*100:.2f}%)")
    print(f"Negative pass      : {negative_passed}/{len(negative_results)} ({negative_passed/len(negative_results)*100:.2f}%)")
    print(f"Mean turns         : {sum(turns)/len(turns):.2f}")
    print(f"Median turns       : {statistics.median(turns):.0f}")

    human = load_json("d4_human_review_edited.json")
    human_pass = sum(1 for x in human if x.get("passed"))

    print("\n" + "=" * 76)
    print("D4 HUMAN JUDGEMENT REVIEW — 9 REPRESENTATIVE CASES")
    print("=" * 76)

    for i, review in enumerate(human, 1):
        case_id = (
            review.get("case_id")
            or review.get("id")
            or review.get("referral_id")
            or f"Review-{i}"
        )

        passed_review = bool(review.get("passed"))

        verdict = (
            review.get("verdict")
            or review.get("judgement")
            or review.get("decision")
            or review.get("rating")
            or ("PASS" if passed_review else "FAIL")
        )

        reason = (
            review.get("reason")
            or review.get("notes")
            or review.get("comment")
            or review.get("rationale")
            or review.get("review")
            or ""
        )

        print(
            f"{case_id:<14} "
            f"{'PASS' if passed_review else 'FAIL':<5} "
            f"verdict={verdict}"
        )

        if reason:
            print(f"  Human review: {reason}")

    print("\nD4 HUMAN REVIEW SUMMARY")
    print(f"Representative cases : {len(human)}")

    if human:
        print(
            f"Human judgement pass : "
            f"{human_pass}/{len(human)} "
            f"({human_pass/len(human)*100:.2f}%)"
        )
    else:
        print("Human judgement pass : no reviews found")

    return results

# ---------------------------------------------------------------------
# D5 SAVED ONLY + D6 ACTUAL CALCULATION
# ---------------------------------------------------------------------

def get_d5_rows():
    rows = []
    for p in sorted(RESULTS.glob("d5_*.json")):
        with open(p, encoding="utf-8") as fh:
            d = json.load(fh)

        s = d["summary"]
        meta = d.get("runner_metadata", {})
        model = meta.get("model")
        version = meta.get("version")

        if not model:
            stem = p.stem
            if "gpt-4o-mini_v1" in stem:
                model = "openai/gpt-4o-mini"
                version = "v1"
            else:
                model = stem

        rows.append({
            "file": p.name,
            "model": model,
            "version": version or ("v1" if "_v1_" in p.name else "v2"),
            "passed": s["passed"],
            "trials": s["trials"],
            "pass_rate": s["pass_rate"],
            "negative_pass_rate": s["negative_pass_rate"],
            "tokens_in": s["tokens_in"],
            "tokens_out": s["tokens_out"],
            "cost_usd": s["cost_usd"],
            "mean_turns": s["mean_turns"],
        })
    return rows

def show_d5_saved_and_run_d6():
    rows = get_d5_rows()

    # ================================================================
    # D5 — LIVE MODEL RESULTS
    # ================================================================
    print("=" * 88)
    print("D5 — SAVED LIVE-MODEL RESULTS (NO NEW API CALLS)")
    print("=" * 88)

    v2_rows = [r for r in rows if r["version"] == "v2"]
    v1_rows = [r for r in rows if r["version"] == "v1"]

    for r in v2_rows:
        print(
            f"{r['model']:<38} "
            f"{r['passed']:>2}/{r['trials']:<2} "
            f"{r['pass_rate'] * 100:>6.2f}%  "
            f"neg {r['negative_pass_rate'] * 100:>6.2f}%"
        )

    if v1_rows:
        r = v1_rows[0]
        print(
            f"\nControlled v1: {r['model']} — "
            f"{r['passed']}/{r['trials']} "
            f"({r['pass_rate'] * 100:.2f}%)"
        )

    # ================================================================
    # D6 — COST TO SERVE
    # ================================================================
    print("\n" + "=" * 100)
    print("D6 — COST TO SERVE ACROSS ALL FIVE V2 MODELS")
    print("=" * 100)

    MONTHLY_VOLUME = 4000
    FALLBACK_COST = 9.17

    # Prices already used in your D6 comparison.
    # Other models will use pricing metadata from their saved D5 JSON
    # when available.
    known_prices = {
        "google/gemini-2.5-flash-lite": {
            "input": 0.10,
            "output": 0.40,
        },
        "openai/gpt-5-mini": {
            "input": 0.25,
            "output": 2.00,
        },
    }

    def get_saved_prices(row):
        """
        Get list-price metadata from the saved D5 JSON.
        Falls back to known project prices for Gemini and GPT-5 Mini.
        """
        model = row["model"]

        if model in known_prices:
            return (
                known_prices[model]["input"],
                known_prices[model]["output"],
            )

        try:
            result_path = RESULTS / row["file"]

            with open(result_path, "r", encoding="utf-8") as f:
                raw = json.load(f)

            meta = raw.get("runner_metadata", {})

            price_in = meta.get("price_in_usd_per_1m")
            price_out = meta.get("price_out_usd_per_1m")

            if price_in is not None and price_out is not None:
                return float(price_in), float(price_out)

        except Exception:
            pass

        return None, None

    def calculate_cost(row):
        price_in, price_out = get_saved_prices(row)

        if price_in is None or price_out is None:
            return None

        mean_input_tokens = row["tokens_in"] / row["trials"]
        mean_output_tokens = row["tokens_out"] / row["trials"]

        # Layer 1 — model/token cost
        layer1 = (
            mean_input_tokens / 1_000_000 * price_in
            + mean_output_tokens / 1_000_000 * price_out
        )

        # Layer 2 — expected human fallback
        layer2 = (
            1 - row["pass_rate"]
        ) * FALLBACK_COST

        total = layer1 + layer2
        monthly = total * MONTHLY_VOLUME

        return {
            "price_in": price_in,
            "price_out": price_out,
            "layer1": layer1,
            "layer2": layer2,
            "total": total,
            "monthly": monthly,
        }

    cost_rows = []

    for row in v2_rows:
        cost = calculate_cost(row)

        combined = dict(row)

        if cost is not None:
            combined.update(cost)

        cost_rows.append(combined)

    # ------------------------------------------------
    # Show all five models
    # ------------------------------------------------
    print(
        f"{'Model':<38}"
        f"{'Success':>10}"
        f"{'Layer 1':>13}"
        f"{'Layer 2':>13}"
        f"{'Total/ref':>13}"
        f"{'Monthly':>14}"
    )

    print("-" * 101)

    for r in cost_rows:
        if "total" not in r:
            print(
                f"{r['model']:<38}"
                f"{r['pass_rate'] * 100:>9.2f}%"
                f"{'price metadata unavailable':>53}"
            )
        else:
            print(
                f"{r['model']:<38}"
                f"{r['pass_rate'] * 100:>9.2f}%"
                f"${r['layer1']:>11.6f}"
                f"${r['layer2']:>11.6f}"
                f"${r['total']:>11.6f}"
                f"${r['monthly']:>12,.2f}"
            )

    print("\nCost assumptions:")
    print(f"  Monthly volume      : {MONTHLY_VOLUME:,} referrals")
    print(f"  Human fallback      : ${FALLBACK_COST:.2f} per failed referral")
    print("  Layer 1             : measured tokens × model list price")
    print("  Layer 2             : (1 - measured success rate) × fallback cost")
    print("  Layer 3             : production infrastructure not measured")

    # ================================================================
    # D6 — BREAK-EVEN
    # ================================================================
    print("\n" + "=" * 100)
    print("D6 — BREAK-EVEN: GEMINI 2.5 FLASH LITE vs GPT-5 MINI")
    print("=" * 100)

    def find_model(name):
        for r in cost_rows:
            if name.lower() in r["model"].lower():
                return r
        return None

    gemini = find_model("gemini-2.5-flash-lite")
    gpt5 = find_model("gpt-5-mini")

    if (
        gemini is not None
        and gpt5 is not None
        and "layer1" in gemini
        and "total" in gpt5
    ):
        # C = cheap model token-only cost
        C = gemini["layer1"]

        # E = expensive model all-in cost
        E = gpt5["total"]

        # F = human fallback cost
        F = FALLBACK_COST

        break_even = 1 - ((E - C) / F)

        print(
            f"Gemini Layer 1 model cost : "
            f"${gemini['layer1']:.6f}/referral"
        )
        print(
            f"Gemini Layer 2 fallback   : "
            f"${gemini['layer2']:.6f}/referral"
        )
        print(
            f"Gemini total              : "
            f"${gemini['total']:.6f}/referral"
        )
        print(
            f"Gemini monthly @4,000     : "
            f"${gemini['monthly']:,.2f}"
        )

        print(
            f"GPT-5 Mini total          : "
            f"${gpt5['total']:.6f}/referral"
        )
        print(
            f"GPT-5 Mini monthly        : "
            f"${gpt5['monthly']:,.2f}"
        )

        print(
            f"Gemini break-even success : "
            f"{break_even * 100:.3f}%"
        )
        print(
            f"Gemini measured success   : "
            f"{gemini['pass_rate'] * 100:.3f}%"
        )

        if gemini["pass_rate"] >= break_even:
            print(
                "Conclusion                  : "
                "Gemini measured success is above break-even."
            )
        else:
            print(
                "Conclusion                  : "
                "Gemini measured success is below break-even."
            )

    else:
        print(
            "Break-even calculation unavailable because "
            "required pricing data is missing."
        )

    print("\nShipping caps: 8 turns; 60,000 tokens/run")


# ---------------------------------------------------------------------
# D7 — ACTUAL REPRODUCTION
# ---------------------------------------------------------------------

def _run_subprocess_script(filename):
    env = os.environ.copy()
    env["A2_DATA"] = str(REFERENCE_DATA)
    proc = subprocess.run(
        [sys.executable, filename],
        cwd=str(SCAFFOLD),
        env=env,
        capture_output=True,
        text=True,
    )
    output = (proc.stdout or "") + (proc.stderr or "")
    return proc.returncode, output

def _important_d7_lines(output):
    """Keep the video readable while still executing the full D7 script."""
    keep_terms = [
        "BASELINE", "FIX OFF", "FIX ON", "REGRESSION",
        "CODE CHECK", "median turns", "worst-case turns",
        "turns ", "decision:", "stopped_by", "guardrails fired",
        "PASS -", "pass rate:", "RIGHT answer", "the RIGHT answer",
        "days", "REJECTED", "REF-5602", "cost US$"
    ]
    lines = []
    for line in output.splitlines():
        if any(term.lower() in line.lower() for term in keep_terms):
            lines.append(line)
    return "\n".join(lines)

def run_d7_actual():
    print("=" * 72)
    print("D7 — ACTUAL FAILURE REPRODUCTION")
    print("=" * 72)

    rc1, out1 = _run_subprocess_script("run_failure1.py")
    print("\nFailure 1 — loop control")
    print(_important_d7_lines(out1))
    if rc1 != 0:
        print(out1)
        raise RuntimeError("run_failure1.py failed")

    rc2, out2 = _run_subprocess_script("run_failure2.py")
    print("\nFailure 2 — required urgency-band interface")
    print(_important_d7_lines(out2))
    if rc2 != 0:
        print(out2)
        raise RuntimeError("run_failure2.py failed")

    return out1, out2


def final_screen():
    print("=" * 72)
    print("PE6201 A2 — FINAL DEMO COMPLETE")
    print("=" * 72)
    print("✓ Final scripted agent executed")
    print("✓ Required negative case executed live")
    print("✓ D2 re-executed from submitted code")
    print("✓ D3 guardrails re-executed")
    print("✓ D4 56-trial battery re-executed")
    print("✓ D5 preserved live results displayed without new API calls")
    print("✓ D6 cost formula recomputed from saved D5 measurements")
    print("✓ D7 both failures reproduced from broken code")
    print("=" * 72)

## PRE-RECORD VERIFICATION — run before pressing Record

This checks the final repository state and evidence files. You may leave the output visible when recording starts.

**Do not spend video time waiting for Drive mounting or troubleshooting.**

In [3]:
preflight()

FINAL SUBMISSION PREFLIGHT
PASS  agent.py
PASS  backends.py
PASS  config.py
PASS  guardrails.py
PASS  harness.py
PASS  prompt.py
PASS  run_eval.py
PASS  tools.py
PASS  tools_ori.py
PASS  test_guardrails.py
PASS  agent_broken.py
PASS  tools_broken.py
PASS  run_failure1.py
PASS  run_failure2.py

Result evidence:
PASS  d2a_tool_block_comparison.json
PASS  d2b_comparison_summary.json
PASS  d2c_comparison_summary.json
PASS  d3_guardrail_summary.json
PASS  d4_scripted_56_summary.json
PASS  d4_human_review_edited.json
PASS  d7_failure1_step_cap.json
PASS  d7_failure2_interface_band.json
PASS  D6 Cost-to-Serve Analysis.pdf

D5 result files found: 6
   d5_Chanchai_Chan_openai_gpt-4o-mini_v2_parallel.json
   d5_He_Yujie_openai_gpt-5-mini_v2_parallel.json
   d5_Kenan_openai_gpt-4o-mini_v1_parallel.json
   d5_LIN_ZHIXUN_qwen_qwen-2.5-72b-instruct_v2_parallel.json
   d5_Pan_Junkun_anthropic_claude-3-haiku_v2_parallel.json
   d5_ULFA_google_gemini-2.5-flash-lite_v2_parallel.json

Default backend: sc

True

# START HERE

---

## System + ordinary cases


In [18]:
normal_record, normal_pass, _ = run_case_demo("REF-EV003", verbose=True)
normal_record, normal_pass, _ = run_case_demo("REF-EV032", verbose=True)


EVALUATION CASE
Case ID             : REF-EV003
Patient ID          : P-EV003
Specialty           : CARD
Date received       : 2026-09-09
Clinical summary    : Breathlessness worsening over days with ankle swelling. Query heart failure.
Tests attached      : ['ECG-12', 'BNP-01']

EXPECTED OUTCOME
Expected decision   : book
Expected booking    : CARD-C1 | 2026-09-16 | 08:30
Must record         : urgency band urgent; mandatory pre-referral tests satisfied for CARD; no future appointment in the same specialty; booking uses a free urgent slot inside the legal window

------------------------------------------------------------------------
NOW RUNNING THE AGENT
------------------------------------------------------------------------
  turn 1    · Fetch the referral first; every later call depends on it.
       get_referral               -> {'referral_id': 'REF-EV003', 'patient_id': 'P-EV003', 'referrin…
  turn 2    · Criteria and patient history are both required and independent, so run th

## Negative case

In [19]:
negative_record, negative_pass, _ = run_case_demo("REF-EV035", verbose=True)


EVALUATION CASE
Case ID             : REF-EV035
Patient ID          : P-EV035
Specialty           : CARD
Date received       : 2026-09-09
Clinical summary    : Crushing chest pain during routine activity. Cardiac assessment requested.
Tests attached      : ['ECG-12', 'BNP-01']

EXPECTED OUTCOME
Expected decision   : escalate
Expected trigger    : red_flag_term
Must record         : the red-flag term "crushing chest pain"; escalate_to triage nurse; no slot booked

------------------------------------------------------------------------
NOW RUNNING THE AGENT
------------------------------------------------------------------------
  turn 1    · Fetch the referral first; every later call depends on it.
       get_referral               -> {'referral_id': 'REF-EV035', 'patient_id': 'P-EV035', 'referrin…
  turn 2    · Criteria and patient history are both required and independent, so run them together.
       check_referral_criteria    -> {'red_flag_term': 'crushing chest pain', 'right_depa

## D2 detailed before/after comparisons

In [20]:
d2a_actual = run_d2a_detailed()
print()
d2b_actual = run_d2b_detailed()
print()
d2c_actual = run_d2c_detailed()

D2(a) — TOOL SET MINIMISATION

D2(a) — TOOL SET DIFFERENCE

ORIGINAL — tools_ori.py
  1. get_referral
  2. lookup_patient
  3. check_referral_criteria
  4. get_clinic_slots
  5. book_slot
  6. as_of

FINAL — tools.py
  1. get_referral
  2. lookup_patient
  3. check_referral_criteria
  4. get_clinic_slots
  5. book_slot

CHANGES
  Removed : as_of
  Added   : None
  Kept    : book_slot, check_referral_criteria, get_clinic_slots, get_referral, lookup_patient

WHY
  as_of was removed from the model-facing tool set. Its date/reference-window logic is handled inside check_referral_criteria, so the model does not need a separate call just to obtain the reference date.

RESULT: 6 model-facing tools → 5 model-facing tools

----------------------------------------------------------------------------
BEFORE — tools_ori.py
----------------------------------------------------------------------------
Callable Problem B tools: 6

[1] get_referral
{
  "name": "get_referral",
  "purpose": "Fetch the re

## D3 guardrails + D4 evaluation

In [22]:
d3_actual = run_d3_actual()
print()
d4_actual = run_d4_detailed()

D3 — GUARDRAIL RESULTS
PASS GR-01  step_cap
PASS GR-02  budget_ceiling
PASS GR-03  duplicate_action
PASS GR-04  gate_held
PASS GR-05  gate_held
PASS GR-06  allowed
PASS GR-07  allowed
PASS GR-08  duplicate_action
PASS GR-09  step_cap
PASS GR-10  budget_ceiling
PASS GR-11  duplicate_action
PASS GR-12  budget_ceiling
PASS GR-13  gate_held
PASS GR-14  allowed

D3 SUMMARY
All tests    : 14/14 PASS
Hostile text : 3/3 PASS

D4 — 40 UNIQUE EVALUATION CASES
REF-EV001    ordinary PASS  decision=book                 turns=4
REF-EV002    ordinary PASS  decision=book                 turns=4
REF-EV003    ordinary PASS  decision=book                 turns=4
REF-EV004    ordinary PASS  decision=book                 turns=4
REF-EV005    ordinary PASS  decision=book                 turns=4
REF-EV006    ordinary PASS  decision=book                 turns=4
REF-EV007    ordinary PASS  decision=book                 turns=4
REF-EV008    ordinary PASS  decision=book                 turns=4
REF-EV009    ordin

## D5 saved live models + D6 recomputation

In [23]:
show_d5_saved_and_run_d6()

D5 — SAVED LIVE-MODEL RESULTS (NO NEW API CALLS)
openai/gpt-4o-mini                     47/56  83.93%  neg  62.50%
openai/gpt-5-mini                      54/56  96.43%  neg  91.67%
qwen/qwen-2.5-72b-instruct             53/56  94.64%  neg  87.50%
anthropic/claude-3-haiku               26/56  46.43%  neg  50.00%
google/gemini-2.5-flash-lite           55/56  98.21%  neg  95.83%

Controlled v1: openai/gpt-4o-mini — 29/56 (51.79%)

D6 — COST TO SERVE ACROSS ALL FIVE V2 MODELS
Model                                    Success      Layer 1      Layer 2    Total/ref       Monthly
-----------------------------------------------------------------------------------------------------
openai/gpt-4o-mini                        83.93%$   0.002103$   1.473750$   1.475853$    5,903.41
openai/gpt-5-mini                         96.43%$   0.004580$   0.327500$   0.332080$    1,328.32
qwen/qwen-2.5-72b-instruct                94.64%$   0.003171$   0.491250$   0.494421$    1,977.68
anthropic/claude-3-haiku 

## D7 reproduced failures

In [26]:
d7_failure1_output, d7_failure2_output = run_d7_actual()
print()
final_screen()

D7 — ACTUAL FAILURE REPRODUCTION

Failure 1 — loop control
BACKEND=scripted  FREE, deterministic  |  PROBLEM=B  |  model=(no model)  |  VERSION=v2  |  cap=8 turns  |  autonomy=confirm
  BASELINE (Phase 2 Step 1) - working agent, frozen 40-case battery
  40 of 40 cases passed the code check
  median turns = 4.0   worst-case turns = 4   ->  data-driven cap = 5
  FIX OFF - agent_broken.run_case() (step cap deleted)
  turns 8 · tool calls 9 · tokens 65880 · cost US$0.00691
  decision: 'escalate'    stopped_by: 'budget_ceiling'
  guardrails fired: ['budget_ceiling']
  CODE CHECK against REF-5602's real answer key: FAIL
  FIX ON - agent.run_case() (unmodified), cap = 5 (data-driven)
  turns 6 · tool calls 6 · tokens 33120 · cost US$0.00353
  decision: 'escalate'    stopped_by: 'step_cap'
  PASS - stopped LOUDLY with reason 'step_cap', not a silent empty answer.
  REGRESSION - the cap must not truncate the 40-case battery
  pass rate: 40/40 (was 40/40 before the cap change)
  PASS - worst leg

### Before recording
1. Run Setup, Helper Functions and Preflight.
2. Confirm `tools_ori.py` exists in `A2_scaffold/`.